In [ ]:
!pip install sentence-transformers datasets chromadb unstructured transformers Chroma tiktoken langchain==0.1.1  huggingface_hub


In [ ]:
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

from langchain.document_loaders import TextLoader
from langchain.schema.document import Document
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
MODEL_MINILM = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_MINILM_L12 = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODEL_distiluse= "sentence-transformers/distiluse-base-multilingual-cased-v2"
SAMPLE = 500
dim = 512

# model_id = "aubmindlab/bert-large-arabertv2"
# dim = 1024
# dim = 384
device = "cpu"

In [ ]:

ar_dataset = open("/content/drive/MyDrive/NLP/NLU/NLU/data/semantic_search/ar.txt").readlines()
en_dataset = open("/content/drive/MyDrive/NLP/NLU/NLU/data/semantic_search/en.txt").readlines()


In [ ]:
## Search
import pandas as pd
# test_questions = {
#     "question":[
#         "وا بتعليم أطفالهم القراءة منذ الصغر  يجب على الآباء ",
#         "فالقراءة مفيدة جداً في تنمية شخصية .",
#         "ان القراءة تساعد علي تنميه المهارات و تشغل الشخص عن المعاصى ",
#         "وهل يمكنني استخدام الزبيب كبديل عن الفياجرا لحل مشكلة ضعف الانتصاب؟	بداية أود"
#     ],
#     "answer":[
#         "والأمهات أن يقوموا بتعليم أطفالهم القراءة منذ الصغر  ",
#         "فالقراءة مفيدة جداً في تنمية شخصية الفرد واتساع مداركه.",
#         "حيث أن استغلال وقت الفراغ في قراءة الكتب النافعة ينمي قدرات ومهارات الفرد، ويشغله عن المعاصي والشهوات.",
#         "هل الزبيب مفيد للجماع؟ وهل يمكنني استخدام الزبيب كبديل عن الفياجرا لحل مشكلة ضعف الانتصاب؟	بداية أود التوضيح أنه لا يوجد ما يؤكد فاعلية الزبيب في علاج الاضطرابات الجنسية أو في تحسين الأداء الجنسي خلال الجماع  لذلك لا يمكن اعتبار الزبيب كبديل عن دواء الفياجرا لعلاج مشكلة ضعف الانتصاب وأنصحك في جميع الأحوال في حال وجود مشكلة جنسية بمراجعة الطبيب للكشف عن سببها لأن بعض مشاكل الضعف الجنسي قد تكون ناجمة عن مشاكل صحية أخرى ويمكن علاجها بكفاءة يجدر الذكر أن الزبيب يحتوي على عدد من الفيتامينات والمعادن التي قد تساهم في تحسين الصحة العامة للجسم وهو ما قد ينعكس على الصحة الجنسية أيضًا  ومن العناصر الغذائية في الزبيب والتي قد تلعب دورًا إيجابيًا في تعزيز الصحة الجنسية ما يلي: الحديد: يساعد الحديد على نقل الأكسجين إلى جميع أنحاء الجسم بما في ذلك الأعضاء التناسلية فيتامينات A وC وE: تلعب هذه الفيتامينات دورًا مهمًا في صحة الجهاز التناسلي بشكل عام الألياف: تساعد الألياف على تحسين عملية الهضم وبالتالي زيادة امتصاص العناصر الغذائية المهمة لسلامة الجهاز التناسلي يمكن تناول الزبيب كجزء من نظام غذائي صحي ومتوازن بالإضافة إلى ممارسة الرياضة بانتظام للاستفادة من فوائده الصحية على الجسم بشكل عام "
#     ],
#     "type-of-question":[
#         "تعليم الابنناء",
#         "القراءه",
#         "اهميه القراءه",
#         "الزبيب في علاج"
#     ]

# }
en_test_questions = {
    "question":[
        "reasons for reading it may be slightly",
        "Reading was a skill we developed as we grew up ",
        "We probably guess words before we read them,",
        "we meet with the best football player in the world right now "
    ],
    "answer":[
        " so our reasons for reading it may be slightly more complex than simply for pleasure",
        "Reading was a skill we developed as we grew up and as we became acquainted with different types of text",
        "We almost certainly predict words before we read them, especially as there are some conventions to a postcard.",
        " we meet with the best football player in the world right now"
    ],
    "type-of-question":[
        "reasons for reading",
        "Reading is a skill",
        "guess words",
        "best football player"
    ]

}
df = pd.DataFrame(en_test_questions)
df.head()

,question,answer,type-of-question
0,reasons for reading it may be slightly,so our reasons for reading it may be slightly...,reasons for reading
1,Reading was a skill we developed as we grew up,Reading was a skill we developed as we grew up...,Reading is a skill
2,"We probably guess words before we read them,",We almost certainly predict words before we re...,guess words
3,we meet with the best football player in the w...,we meet with the best football player in the ...,best football player


In [ ]:
en_dataset= "".join(en_dataset)

In [ ]:
distiluse = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def process_and_embed_docs(embeddings, data, model_name):
  loaded_docs = [Document(page_content=data, metadata={'source': 'local'})]
  splitter = RecursiveCharacterTextSplitter(
      separators = [". ", ".", "\n\n", "\n", " ", ""],
      chunk_size=200, chunk_overlap=0
  )
  split_docs = splitter.split_documents(loaded_docs)
  database = Chroma.from_documents(
      documents=split_docs, embedding=embeddings,
      collection_metadata={"hnsw:space": "cosine"}
  )

  return database
def semantic_search(embeddings, model_name, input_text: str,  query: str):
        db = process_and_embed_docs(embeddings, input_text, model_name)

        retrieved_docs = db.similarity_search_with_relevance_scores(query)
        print(retrieved_docs)
        results = []
        matches = []
        for d in retrieved_docs:
            match = d[0].page_content
            sim = d[1]
            if sim >= 0.6:
                start = input_text.index(match)
                end = start + len(match)
                if match not in results:
                    results.append(match)
                    matches.append({"result": match, "start": start, "end": end, "similarity": sim})

        return {"query": query, "matches": matches}

In [ ]:
results_distiluse = []
for question in df['question']:
  print(question)
  results_distiluse.append(semantic_search(distiluse, "./distiluse",en_dataset,  question) )

reasons for reading it may be slightly
[(Document(page_content='.\n\nWhy do we read?\nThere are a number of reasons why we read, and this will often influence what we read and how we read it', metadata={'source': 'local'}), 0.6147994995117188), (Document(page_content='. We could also be reading the lyrics to a song, so our reasons for reading it may be slightly more complex than simply for pleasure', metadata={'source': 'local'}), 0.49676305055618286), (Document(page_content='.\nRationale: Identifying the type of text and where you might read it supplies the reader with some context', metadata={'source': 'local'}), 0.4438976049423218), (Document(page_content='What is reading?\nAt the most basic level reading is the recognition of words', metadata={'source': 'local'}), 0.41316938400268555)]
Reading was a skill we developed as we grew up 
[(Document(page_content='. Reading was a skill we developed as we grew up and as we became acquainted with different types of text. Once we start seein

In [ ]:
df_results  = pd.read_csv("/content/drive/MyDrive/NLP/NLU/NLU/data/semantic_search/en-results.csv")

In [ ]:
df_results.head()

,questions,answers,MINILM_L12
0,reasons for reading it may be slightly,so our reasons for reading it may be slightly...,{'query': 'reasons for reading it may be sligh...
1,Reading was a skill we developed as we grew up,Reading was a skill we developed as we grew up...,{'query': 'Reading was a skill we developed as...
2,"We probably guess words before we read them,",We almost certainly predict words before we re...,{'query': 'We probably guess words before we r...
3,we meet with the best football player in the w...,we meet with the best football player in the ...,{'query': 'we meet with the best football play...


In [ ]:
df_results = df_results.drop("distiluse",axis=1)

In [ ]:
df_results['distiluse'] = results_distiluse

In [ ]:

df_results.to_csv("/content/drive/MyDrive/NLP/NLU/NLU/data/semantic_search/en-results.csv", index=False)